# **Annotate a molecular dynamics description with LLM 📝**

### 🎯 Objectives:

- Annotate a description of a datasats or a Materials and Methods section of a molecular dynamics study.
- Identify key experimental procedures, parameters, and computational tools like MOLECULE (`MOL`), FORCEFIELD OR MODELS (`FFM`), SIMULATION_TIME (`STIME`), TEMPERATURE (`TEMP`), SOFTWARE NAME (`SOFTNAME`) and SOFTWARE VERSION (`SOFTVERS`).
- Control the output format of the annotations with `instructor` framework to reduce hallucinations.
- Visualize the annotation results for clarity and verification.


## **Load libraries and setup**

In [9]:
import os

from dotenv import load_dotenv

from mdner_llm.annotations.visualize_annotations import visualize_llm_annotation
from mdner_llm.core.extract_entities_with_llm import (
    annotate_with_instructor,
    load_prompt,
)
from mdner_llm.logger import create_logger
from mdner_llm.models.entities import ListOfEntities


In [10]:
%load_ext watermark
%watermark

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Last updated: 2026-04-21T17:16:21.954353+02:00

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 8.13.2

Compiler    : Clang 22.1.1 
OS          : Darwin
Release     : 24.0.0
Machine     : arm64
Processor   : arm
CPU cores   : 8
Architecture: 64bit



## **Step 1: Define the text to annotate**


In [11]:
TEXT_TO_ANNOTATE = """Molecular Dynamics Simulations

MD simulations were performed using GROMACS 4.0.7 software [17] with the OPLS-AA
force-field [18].L33 and P33 forms of β3 were soaked in a rhombic dodecahedral
simulation box with 60,622 TIP3P water molecules and 28 Na+ ions.
The distance between any atom of the protein and the box edges was set to at least 10 Å.
The total energy of the system was minimized twice (before and after the addition of the
ions) with a steepest descent algorithm. MD simulations were run under the NPT
thermodynamic ensemble and periodic boundary conditions were applied in all directions.
We used the weak coupling algorithm of Berendsen [19] to maintain the system at a
constant physiological temperature of 310 K using a coupling constant of 0.1 ps (protein
and water ions separately). Pressure was held constant using the Berendsen algorithm
[19] at 1 atm with a coupling constant of 1 ps. Water molecules were kept rigid using
the SETTLE algorithm [20]. All other bond lengths were constrained with the LINCS
algorithm [21], allowing a 2 fs time step. We used a short-range coulombic and van der
Waals cut-off of 10 Å and calculated the long-range electrostatic interactions using
the smooth particle mesh Ewald (PME) algorithm [22], [23] with a grid spacing of 1.2 Å
and an interpolation order of 4. The neighbor list was updated every 10 steps. After a
1 ns equilibration (with position restraints on the protein), each system was simulated
for 50 ns. For both systems, five 50 ns simulations were performed (using different
initial velocities) and one was extended until 100 ns for a total simulation time of
300 ns. Molecular conformations were saved every 100 ps for further analysis.
"""

## **Step 2: Select the model**


But before, we retrieve the API key for the openrouter API from the environment variables. Make sure to set the `OPENROUTER_API_KEY` environment variable before running this cell.


In [12]:
# Set the openrouter API key
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

In [13]:
# Choices:
# Docs: https://openrouter.ai/models
# We will use "openai/gpt-5.2" for annotation
SELECTED_MODEL = "openai/gpt-5.2"

## **Step 3: Load the prompt template**


In [14]:
prompt_file = "json_few_shot.txt"
prompt = load_prompt(
    prompt_file=prompt_file,
    logger=create_logger(),
)
print(f"Prompt loaded from {prompt_file}:")
print(f"{prompt[:200]}...")

Prompt loaded from json_few_shot.txt:
# Named-Entity Recognition task

## Role definition

You are a highly specialized **Named Entity Recognition (NER) engine** dedicated to extracting entities from Molecular Dynamics (MD) simulation dat...


## **Step 4: Annotate the text**


In [15]:
# Define the system and user messages for the LLM prompt
system_msg = "Extract entities as structured JSON."
user_msg = f"{prompt}\n{TEXT_TO_ANNOTATE}"
messages = [
    {"role": "system", "content": system_msg},
    {"role": "user", "content": user_msg},
]
# Initialize metadata for tracking LLM response and usage
metadata = {
    "raw_llm_response": None,
    "usage": {
        "inference_time_sec": None,
        "cost": None,
        "input_tokens": None,
        "output_tokens": None,
    },
    "status": None,
}
response, inference_metadata = annotate_with_instructor(
    SELECTED_MODEL,
    API_KEY,
    metadata,
    messages,
    response_model=ListOfEntities,
)
print(
    f"Annotation by {SELECTED_MODEL} completed in "
    f"{inference_metadata['usage']['inference_time_sec']:.2} "
    f"seconds with cost ${inference_metadata['usage']['cost']:.4f}."
)
response

Annotation by openai/gpt-5.2 completed in 5.6 seconds with cost $0.0045.


ListOfEntities(entities=[SoftwareName(category='SOFTNAME', text='GROMACS'), SoftwareVersion(category='SOFTVERS', text='4.0.7'), ForceField(category='FFM', text='OPLS-AA'), Molecule(category='MOL', text='L33'), Molecule(category='MOL', text='P33'), Molecule(category='MOL', text='β3'), ForceField(category='FFM', text='TIP3P'), Molecule(category='MOL', text='Na+'), SoftwareName(category='SOFTNAME', text='Berendsen'), Temperature(category='TEMP', text='310 K'), SoftwareName(category='SOFTNAME', text='SETTLE'), SoftwareName(category='SOFTNAME', text='LINCS'), SoftwareName(category='SOFTNAME', text='Ewald'), SoftwareName(category='SOFTNAME', text='PME'), SimulationTime(category='STIME', text='0.1 ps'), SimulationTime(category='STIME', text='1 ps'), SimulationTime(category='STIME', text='2 fs'), SimulationTime(category='STIME', text='1 ns'), SimulationTime(category='STIME', text='50 ns'), SimulationTime(category='STIME', text='50 ns'), SimulationTime(category='STIME', text='100 ns'), Simulati

## Step 5: Visualize the annotation results


In [16]:
visualize_llm_annotation(response, TEXT_TO_ANNOTATE)

🧐 VISUALIZATION OF ENTITIES 
